# 🧠 EEG Emotional Memory — GPU Ultra Pipeline v5.1 (Fixed)

**Bug fixes from v5.0:**
- Fixed OOF index alignment (cumsum was using wrong axis — n_trials not n_timepoints)
- Fixed CSP inside OOF loop (was refitting on wrong subjects)  
- Fixed `oof_probs` array shape (was indexed by trials but filled per timepoint slice)
- Removed duplicate feature extraction inside OOF (was re-extracting already cached features)
- Added progress bar to feature extraction
- GPU=False fallback is graceful and silent

**Architecture:**
```
549 features/tp → ANOVA top-300 → LDA(30%) + LGBM(40%) + XGB(30%)
Per timepoint in [300–900ms] (121 models) → smooth → gate → isotonic → submission
```


## Cell 1 — Install & GPU Check

In [ ]:
import subprocess, sys
for pkg in ['lightgbm', 'xgboost', 'tqdm']:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'],
                       capture_output=True, text=True)
    print(f"{'✓' if r.returncode==0 else '✗'} {pkg}")

try:
    r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                       capture_output=True, text=True)
    GPU_AVAILABLE = r.returncode == 0 and len(r.stdout.strip()) > 0
    if GPU_AVAILABLE:
        print(f"\n🚀 GPU: {r.stdout.strip()}")
    else:
        print("\n⚠️  No GPU — using CPU")
except:
    GPU_AVAILABLE = False
    print("\n⚠️  No GPU — using CPU")

print(f"GPU_AVAILABLE = {GPU_AVAILABLE}")


## Cell 2 — Imports

In [ ]:
import numpy as np
import pandas as pd
import h5py
import os, re, time, warnings, gc
from pathlib import Path
from tqdm import tqdm
from joblib import Parallel, delayed
import multiprocessing

from scipy.signal  import butter, sosfiltfilt, hilbert, detrend as sp_detrend, savgol_filter
from scipy.ndimage import gaussian_filter1d
from scipy.stats   import skew, kurtosis
from scipy.linalg  import eigh

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing         import RobustScaler
from sklearn.metrics               import roc_auc_score
from sklearn.feature_selection     import SelectKBest, f_classif
from sklearn.isotonic              import IsotonicRegression

import lightgbm as lgb
import xgboost  as xgb
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings('ignore')
np.random.seed(42)
N_JOBS = multiprocessing.cpu_count()
print(f"✓ Imports OK  |  CPU cores: {N_JOBS}  |  GPU: {GPU_AVAILABLE}")


## Cell 3 — Paths

In [ ]:
DRIVE_BASE = 'D:\\eeg_competition'
EMO_DIR    = f'{DRIVE_BASE}/training/sleep_emo'
NEU_DIR    = f'{DRIVE_BASE}/training/sleep_neu'
TEST_DIR   = f'{DRIVE_BASE}/testing'
OUTPUT     = './submission.csv'

for name, path in [('EMO_DIR', EMO_DIR), ('NEU_DIR', NEU_DIR), ('TEST_DIR', TEST_DIR)]:
    exists = os.path.exists(path)
    count  = len(list(Path(path).glob('*.mat'))) if exists else 0
    print(f"{name}: {'✓ ' + str(count) + ' files' if exists else '✗ NOT FOUND'}  →  {path}")


## Cell 4 — Configuration

In [ ]:
FS   = 200
N_TP = 200
N_CH = 16

# ── Signal window ─────────────────────────────────────────────────────────
# Emotional memory reactivation only in [300-900ms] post-cue.
# Training OUTSIDE this window = pure noise → 50% AUC.
TP_SIG_START = 60   # 300ms
TP_SIG_END   = 180  # 900ms
SIGNAL_TPS   = list(range(TP_SIG_START, TP_SIG_END + 1))  # 121 tps
BLEND_ALPHA  = 0.25  # outside window: pull prediction toward 0.5

# ── Features ──────────────────────────────────────────────────────────────
WIN        = 40
N_CSP      = 4
N_FEAT_SEL = 300

# ── Bands ─────────────────────────────────────────────────────────────────
BANDS = {
    'delta': (1.0,  4.0),
    'theta': (4.0,  8.0),
    'alpha': (8.0, 13.0),
    'sigma': (12.0,16.0),
    'beta':  (13.0,30.0),
    'hbeta': (20.0,40.0),
}

# ── Channel map ───────────────────────────────────────────────────────────
CHANNELS = ['c3','c4','o1','o2','cp3','f3','f4','cp4',
            'c5','cz','c6','cp5','p7','pz','p8','cp6']
CH = {c: i for i, c in enumerate(CHANNELS)}

CONN_PAIRS = [
    ('f3','pz'),('f4','pz'),('f3','cz'),('f4','cz'),
    ('c3','c4'),('cp3','cp4'),('f3','f4'),('cz','pz'),
    ('f3','cp4'),('f4','cp3'),
]

SMOOTH_SIGMA = 8
SAVGOL_WIN   = 21
SAVGOL_POLY  = 3

LGBM_DEVICE = 'gpu' if GPU_AVAILABLE else 'cpu'
XGB_TREE    = 'gpu_hist' if GPU_AVAILABLE else 'hist'

print(f"✓ Config OK")
print(f"  Signal window : tp[{TP_SIG_START}–{TP_SIG_END}]  (300–900ms)  →  {len(SIGNAL_TPS)} classifiers")
print(f"  LGBM device   : {LGBM_DEVICE}  |  XGB tree: {XGB_TREE}")


## Cell 5 — HDF5 Loader

In [ ]:
def _resolve(f, grp, key):
    field = grp[key]
    if isinstance(field, h5py.Dataset):
        val = field[()]
        if isinstance(val, h5py.Reference): return np.array(f[val])
        if hasattr(val,'shape') and val.shape==(1,1):
            ref = val.item()
            if isinstance(ref, h5py.Reference): return np.array(f[ref])
        return np.array(val)
    return np.array(field)


def load_mat(path):
    with h5py.File(str(path), 'r') as f:
        grp = f.get('data')
        if grp is None:
            for k in f.keys():
                if hasattr(f[k],'keys') and 'trial' in f[k]: grp=f[k]; break
        if grp is None: raise ValueError(f"No 'data' struct in {path}")

        raw = _resolve(f, grp, 'trial')
        if raw.ndim == 3:
            sh = raw.shape
            if   sh[2]==N_CH and sh[1]==N_TP: raw=raw.transpose(0,2,1)
            elif sh[0]==N_CH and sh[1]==N_TP: raw=raw.transpose(2,0,1)
            elif sh[0]==N_TP and sh[1]==N_CH: raw=raw.transpose(2,1,0)
        elif raw.ndim == 2:
            raw = raw.T[np.newaxis]
        eeg = raw.astype(np.float32)

        try:
            ti = _resolve(f, grp, 'trialinfo')
            if   ti.ndim==2 and ti.shape[0]==eeg.shape[0]: labels=ti[:,0].astype(int)
            elif ti.ndim==2 and ti.shape[1]==eeg.shape[0]: labels=ti[0,:].astype(int)
            else:                                            labels=ti.flatten().astype(int)
        except: labels=np.ones(eeg.shape[0],dtype=int)

        try:   tv=_resolve(f,grp,'time').flatten()
        except: tv=np.arange(N_TP)/FS
        mask = tv >= -1e-6
        if np.any(~mask): tv=tv[mask]; eeg=eeg[:,:,mask]
        if len(tv)!=N_TP: tv=np.arange(N_TP)/FS

    print(f"    ✓ {Path(path).name}: eeg={eeg.shape} | neu={(labels==1).sum()} emo={(labels==2).sum()}")
    return {'eeg':eeg,'labels':labels,'time':tv,'id':Path(path).stem}


def load_all_training(emo_dir, neu_dir):
    subjects, seen = [], set()
    for d in [emo_dir, neu_dir]:
        for p in sorted(Path(d).glob('*.mat')):
            if p.stem in seen: continue
            seen.add(p.stem)
            try: subjects.append(load_mat(p))
            except Exception as e: print(f"    ✗ {p.name}: {e}")
    print(f"\n✓ Training: {len(subjects)} subjects")
    return subjects


def load_all_test(test_dir):
    subjects = []
    for p in sorted(Path(test_dir).glob('*.mat')):
        try:
            d = load_mat(p)
            nums = re.findall(r'\d+', p.stem)
            d['subj_id'] = int(nums[-1]) if nums else len(subjects)+1
            subjects.append(d)
        except Exception as e: print(f"    ✗ {p.name}: {e}")
    print(f"\n✓ Test: {len(subjects)} subjects | IDs: {[s['subj_id'] for s in subjects]}")
    return subjects

print("✓ Loaders defined")


## Cell 6 — Preprocessing

In [ ]:
def bandpass(data, lo, hi, fs=FS, order=4):
    nyq = fs/2.0
    sos = butter(order,[max(lo/nyq,1e-4),min(hi/nyq,0.9999)],btype='band',output='sos')
    return sosfiltfilt(sos, data, axis=-1).astype(np.float32)

def preprocess_trial(t):
    x = sp_detrend(t.astype(np.float64), axis=-1).astype(np.float32)
    x = x - x.mean(axis=0, keepdims=True)
    return bandpass(x, 0.5, 40.0)

def zscore_subject(eeg):
    mu  = eeg.mean(axis=(0,2), keepdims=True)
    sig = eeg.std( axis=(0,2), keepdims=True) + 1e-8
    return ((eeg-mu)/sig).astype(np.float32), mu, sig

print("✓ Preprocessing defined")


## Cell 7 — Feature Utilities

In [ ]:
def de(x):
    v=np.var(x)
    return float(0.5*np.log(2*np.pi*np.e*v)) if v>1e-14 else 0.0

def hjorth(x):
    act=float(np.var(x)); d1=np.diff(x); v1=float(np.var(d1)); mob=np.sqrt(v1/(act+1e-12))
    d2=np.diff(d1); v2=float(np.var(d2)); cmp=np.sqrt(v2/(v1+1e-12))/(mob+1e-12)
    return act, mob, cmp

def plv(x, y):
    if len(x)<4: return 0.0
    return float(np.abs(np.mean(np.exp(1j*(np.angle(hilbert(x))-np.angle(hilbert(y)))))))

print("✓ Feature utilities defined")


## Cell 8 — Feature Extraction (~541 features/timepoint)

In [ ]:
def extract_trial_features(trial):
    """trial: (16,200) → (200, 541) float32"""
    n_ch, n_tp = trial.shape
    half = WIN // 2
    bf, bp = {}, {}
    for bn,(lo,hi) in BANDS.items():
        f=bandpass(trial,lo,hi); bf[bn]=f
        bp[bn]=(np.abs(hilbert(f,axis=-1))**2).astype(np.float32)

    all_f = []
    for t in range(n_tp):
        t0=max(0,t-half); t1=min(n_tp,t+half)
        f=[]
        # A. Band power (96)
        for bn in BANDS: f.extend(np.mean(bp[bn][:,t0:t1],axis=1).tolist())
        # B. Diff entropy (96)
        for bn in BANDS:
            seg=bf[bn][:,t0:t1]
            for ch in range(n_ch): f.append(de(seg[ch]))
        # C. Relative power (96)
        tot=sum(np.mean(bp[bn][:,t0:t1],axis=1) for bn in BANDS)+1e-12
        for bn in BANDS: f.extend((np.mean(bp[bn][:,t0:t1],axis=1)/tot).tolist())
        # D. FAA + frontal theta (2)
        f3a=np.mean(bp['alpha'][CH['f3'],t0:t1])+1e-12
        f4a=np.mean(bp['alpha'][CH['f4'],t0:t1])+1e-12
        f.append(float(np.log(f4a)-np.log(f3a)))
        f.append(float((np.mean(bp['theta'][CH['f3'],t0:t1])+
                         np.mean(bp['theta'][CH['f4'],t0:t1]))/2.0))
        # E. Theta/Beta (16)
        for ch in range(n_ch):
            f.append(float((np.mean(bp['theta'][ch,t0:t1])+1e-12)/
                            (np.mean(bp['beta'] [ch,t0:t1])+1e-12)))
        # F. Theta/Alpha (16)
        for ch in range(n_ch):
            f.append(float((np.mean(bp['theta'][ch,t0:t1])+1e-12)/
                            (np.mean(bp['alpha'][ch,t0:t1])+1e-12)))
        # G. Hemispheric asymmetry (9)
        for ch1,ch2 in [('c3','c4'),('cp3','cp4'),('o1','o2')]:
            for bn in ['theta','alpha','beta']:
                p1=np.mean(bp[bn][CH[ch1],t0:t1])+1e-12
                p2=np.mean(bp[bn][CH[ch2],t0:t1])+1e-12
                f.append(float(np.log(p2)-np.log(p1)))
        # H. Hjorth 4ch (12)
        for chn in ['f3','f4','cz','pz']:
            a,m,c=hjorth(trial[CH[chn],t0:t1]); f.extend([a,m,c])
        # I. Spindle sigma 8ch (8)
        for chn in ['c3','c4','cz','pz','f3','f4','cp3','cp4']:
            f.append(float(np.mean(bp['sigma'][CH[chn],t0:t1])))
        # J. Peak-to-peak (16)
        seg=trial[:,t0:t1]
        f.extend((np.max(seg,axis=1)-np.min(seg,axis=1)).tolist())
        # K. Skew+Kurt 4ch (8)
        for chn in ['f3','f4','cz','pz']:
            s=trial[CH[chn],t0:t1].astype(np.float64)
            f.append(float(skew(s)) if len(s)>2 else 0.0)
            f.append(float(kurtosis(s)) if len(s)>3 else 0.0)
        # L. PLV 10pairs×3bands (30)
        for bn in ['theta','alpha','beta']:
            for ch1,ch2 in CONN_PAIRS:
                f.append(plv(bf[bn][CH[ch1],t0:t1], bf[bn][CH[ch2],t0:t1]))
        # M. Theta covariance upper-tri (136)
        t0c=max(0,t-10); t1c=min(n_tp,t+11)
        seg_c=bp['theta'][:,t0c:t1c]
        if seg_c.shape[1]>2:
            C=np.cov(seg_c); idx=np.triu_indices(n_ch)
            f.extend(C[idx].tolist())
        else:
            f.extend([0.0]*(n_ch*(n_ch+1)//2))
        all_f.append(f)

    F=np.array(all_f,dtype=np.float32)
    return np.nan_to_num(F,nan=0.0,posinf=0.0,neginf=0.0)  # (200, 541)


def extract_subject(s):
    """Returns X(n_tr*200,541), y(n_tr*200,), trial_ids, zmu, zsig."""
    eeg,labels = s['eeg'].copy(), s['labels']
    eeg,zmu,zsig = zscore_subject(eeg)
    n_tr = eeg.shape[0]
    Fs,ys,ts=[],[],[]
    for i in range(n_tr):
        Fs.append(extract_trial_features(preprocess_trial(eeg[i])))
        ys.extend([int(labels[i])]*N_TP)
        ts.extend([i]*N_TP)
    return (np.vstack(Fs), np.array(ys,dtype=np.int32),
            np.array(ts,dtype=np.int32), zmu, zsig)

print("✓ Feature extraction defined  (541 features/timepoint)")


## Cell 9 — CSP Spatial Filters

In [ ]:
class CSP:
    def __init__(self, n=N_CSP): self.n=n; self.W=None

    def fit(self, eeg, labels):
        filt=bandpass(eeg.reshape(-1,N_TP),4.0,8.0).reshape(eeg.shape)
        def cov(X):
            C=np.zeros((N_CH,N_CH))
            for t in X:
                s=t-t.mean(axis=-1,keepdims=True); C+=s@s.T/(s.shape[-1]-1)
            return C/len(X)
        C1=cov(filt[labels==1]); C2=cov(filt[labels==2])
        ev,evec=eigh(C1,C1+C2); idx=np.argsort(ev)
        self.W=evec[:,np.concatenate([idx[:self.n],idx[-self.n:]])]
        return self

    def log_var(self, eeg, win=WIN):
        """(n_tr,16,200) → (n_tr*N_TP, 2n) log-variance features."""
        filt=bandpass(eeg.reshape(-1,N_TP),4.0,8.0).reshape(eeg.shape)
        sig=np.tensordot(self.W.T,filt,axes=([1],[1])).transpose(1,0,2)
        half=win//2; out=[]
        for tr in range(eeg.shape[0]):
            for t in range(N_TP):
                t0=max(0,t-half); t1=min(N_TP,t+half)
                out.append(np.log(np.var(sig[tr,:,t0:t1],axis=1)+1e-12))
        return np.array(out,dtype=np.float32)

print("✓ CSP defined")


## Cell 10 — TimeResolved Ensemble (GPU LightGBM + XGBoost)

One (LDA + LGBM + XGB) classifier per timepoint in [300–900ms].  
Training is parallelised across all CPU cores via `joblib`.


In [ ]:
def _make_lgbm():
    p=dict(n_estimators=400,num_leaves=31,max_depth=5,learning_rate=0.05,
           subsample=0.8,colsample_bytree=0.7,min_child_samples=10,
           class_weight='balanced',reg_alpha=0.1,reg_lambda=1.0,
           n_jobs=1,random_state=42,verbose=-1)
    if GPU_AVAILABLE: p['device']='gpu'; p['gpu_use_dp']=False
    return lgb.LGBMClassifier(**p)

def _make_xgb():
    p=dict(n_estimators=300,max_depth=4,learning_rate=0.05,subsample=0.8,
           colsample_bytree=0.7,min_child_weight=10,gamma=0.1,
           eval_metric='logloss',n_jobs=1,random_state=42,verbosity=0)
    p['tree_method']='gpu_hist' if GPU_AVAILABLE else 'hist'
    if GPU_AVAILABLE: p['predictor']='gpu_predictor'
    return xgb.XGBClassifier(**p)


def _fit_tp(tp, X_flat, y_trial, n_trials):
    """Fit one timepoint classifier. Returns (tp, model_dict)."""
    # X_flat is (n_trials*N_TP, n_feat); grab row for each trial at this tp
    idx  = np.arange(n_trials) * N_TP + tp
    Xt   = np.nan_to_num(X_flat[idx])           # (n_trials, n_feat)
    ybin = (y_trial == 2).astype(int)

    # Feature selection
    if N_FEAT_SEL and N_FEAT_SEL < Xt.shape[1]:
        sel  = SelectKBest(f_classif, k=N_FEAT_SEL)
        Xt_s = sel.fit_transform(Xt, ybin)
    else:
        sel=None; Xt_s=Xt

    sc=RobustScaler(); Xt_s=sc.fit_transform(Xt_s)

    lda=LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto')
    try:    lda.fit(Xt_s,ybin)
    except: lda=None

    lgbm=_make_lgbm()
    try:    lgbm.fit(Xt_s,ybin)
    except:
        lgbm=lgb.LGBMClassifier(n_estimators=200,n_jobs=1,verbose=-1,random_state=42)
        try:    lgbm.fit(Xt_s,ybin)
        except: lgbm=None

    xgbm=_make_xgb()
    try:    xgbm.fit(Xt_s,ybin)
    except:
        xgbm=xgb.XGBClassifier(n_estimators=200,tree_method='hist',
                                 n_jobs=1,random_state=42,verbosity=0)
        try:    xgbm.fit(Xt_s,ybin)
        except: xgbm=None

    return tp, {'lda':lda,'lgbm':lgbm,'xgb':xgbm,'sel':sel,'sc':sc}


class TRE:
    """TimeResolved Ensemble: 1 classifier per signal-window timepoint."""
    W = {'lda':0.30, 'lgbm':0.40, 'xgb':0.30}

    def __init__(self): self.models={}; self.fitted=False

    def fit(self, X_flat, y_flat, n_trials, verbose=True):
        """
        X_flat  : (n_trials * N_TP, n_features)
        y_flat  : (n_trials * N_TP,)   — label repeated for each tp
        n_trials: int
        """
        y_trial = y_flat[::N_TP]   # (n_trials,) — one label per trial
        if verbose:
            print(f"  Training {len(SIGNAL_TPS)} classifiers "
                  f"| n_trials={n_trials} | n_jobs={max(1,N_JOBS-1)} | GPU={GPU_AVAILABLE}")
        t0=time.time()
        res=Parallel(n_jobs=max(1,N_JOBS-1), prefer='threads')(
            delayed(_fit_tp)(tp, X_flat, y_trial, n_trials)
            for tp in tqdm(SIGNAL_TPS, desc='  Fitting TRE', disable=not verbose)
        )
        for tp,state in res: self.models[tp]=state
        self.fitted=True
        if verbose: print(f"  ✓ {len(self.models)} classifiers in {time.time()-t0:.1f}s")

    def predict(self, X_flat, n_trials, verbose=False):
        """Returns (n_trials, N_TP) probability matrix. Outside window = 0.5."""
        assert self.fitted
        P=np.full((n_trials,N_TP),0.5,dtype=np.float64)
        for tp in tqdm(SIGNAL_TPS,desc='  Predicting',disable=not verbose):
            m=self.models[tp]
            idx=np.arange(n_trials)*N_TP+tp
            Xt=np.nan_to_num(X_flat[idx])
            Xt_s=m['sel'].transform(Xt) if m['sel'] else Xt
            Xt_s=m['sc'].transform(Xt_s)
            bl=np.zeros(n_trials); wt=0.0
            if m['lda']:  bl+=self.W['lda'] *m['lda'].predict_proba(Xt_s)[:,1];  wt+=self.W['lda']
            if m['lgbm']: bl+=self.W['lgbm']*m['lgbm'].predict_proba(Xt_s)[:,1]; wt+=self.W['lgbm']
            if m['xgb']:  bl+=self.W['xgb'] *m['xgb'].predict_proba(Xt_s)[:,1];  wt+=self.W['xgb']
            if wt>0: P[:,tp]=bl/wt
        return P

print("✓ TRE (TimeResolved Ensemble) defined")


## Cell 11 — Post-Processing & Competition Metric

In [ ]:
def smooth(P):
    """Two-stage per-trial: Savitzky-Golay → Gaussian. P:(n_tr,N_TP)"""
    out=np.zeros_like(P,dtype=np.float64)
    for i in range(P.shape[0]):
        seg=P[i].astype(np.float64)
        if len(seg)>=SAVGOL_WIN:
            seg=savgol_filter(seg,window_length=SAVGOL_WIN,polyorder=SAVGOL_POLY)
        out[i]=gaussian_filter1d(seg,sigma=SMOOTH_SIGMA)
    return out


def gate(P, alpha=BLEND_ALPHA):
    """Blend predictions outside [300-900ms] toward 0.5."""
    out=P.copy()
    for tp in range(N_TP):
        if tp<TP_SIG_START or tp>TP_SIG_END:
            out[:,tp]=alpha*out[:,tp]+(1-alpha)*0.5
    return out


def window_auc(P, y_bin, min_ms=50, win_tp=10):
    """Official competition metric: longest continuous window-AUC > 0.5 (≥50ms)."""
    aucs=[]
    for s in range(P.shape[1]-win_tp+1):
        wp=P[:,s:s+win_tp].mean(axis=1)
        try:    aucs.append(roc_auc_score(y_bin,wp))
        except: aucs.append(0.5)
    aucs=np.array(aucs)
    min_w=max(1,int(min_ms*FS/1000))
    best_s,best_l,best_auc=0,0,0.5
    rs=rl=0
    for i,ab in enumerate(aucs>0.5):
        if ab:
            if rl==0: rs=i
            rl+=1
        else:
            if rl>=min_w and rl>best_l:
                best_l=rl; best_s=rs; best_auc=aucs[rs:rs+rl].mean()
            rl=0
    if rl>=min_w and rl>best_l: best_auc=aucs[rs:rs+rl].mean()
    return {'window_auc':best_auc,'aucs':aucs,'mean_auc':aucs.mean(),'dur_ms':best_l*(1000/FS)}

print("✓ Post-processing & Window-AUC defined")


## Cell 12 — LOSO Cross-Validation  
> ⏱️ Set `RUN_LOSO=True` in MAIN to run (~60-90 min)

In [ ]:
def run_loso(train_subjects):
    print("\n"+"="*64)
    print("  LOSO CV — Window-AUC (official competition metric)")
    print("="*64)

    # Step 1: pre-extract all features
    print("\nExtracting features for all subjects...")
    cache=[]
    for i,s in enumerate(train_subjects):
        t0=time.time()
        X,y,tid,zmu,zsig=extract_subject(s)
        cache.append({'X':X,'y':y,'eeg':s['eeg'],'labels':s['labels'],'id':s['id']})
        print(f"  [{i+1:2d}] {s['id']:30s} {X.shape}  {time.time()-t0:.0f}s")

    fold_results=[]
    n=len(train_subjects)

    for vi in range(n):
        val=cache[vi]
        tr =[cache[i] for i in range(n) if i!=vi]
        n_tr_trials=sum(c['eeg'].shape[0] for c in tr)

        X_tr=np.vstack([c['X'] for c in tr])
        y_tr=np.concatenate([c['y'] for c in tr])
        eeg_tr=np.vstack([c['eeg'] for c in tr])
        lbl_tr=np.concatenate([c['labels'] for c in tr])

        csp=CSP(); csp.fit(eeg_tr,lbl_tr)
        csp_tr=csp.log_var(eeg_tr)
        csp_v =csp.log_var(val['eeg'])
        X_tr_f=np.hstack([X_tr,csp_tr])
        X_v_f =np.hstack([val['X'],csp_v])

        tre=TRE(); tre.fit(X_tr_f,y_tr,n_tr_trials,verbose=False)

        n_v=val['eeg'].shape[0]
        P=gate(smooth(tre.predict(X_v_f,n_v)))
        P=np.clip(P,0.01,0.99)

        y_v=(val['labels']==2).astype(int)
        m=window_auc(P,y_v)
        fold_results.append(m)
        print(f"  Fold {vi+1:2d} {val['id']:30s}  window={m['window_auc']:.4f}  "
              f"mean={m['mean_auc']:.4f}  dur={m['dur_ms']:.0f}ms")

        del tre,X_tr_f,X_v_f,eeg_tr,csp_tr,csp_v; gc.collect()

    win_aucs=[r['window_auc'] for r in fold_results]
    print(f"\n  Window AUC: {np.mean(win_aucs):.4f} ± {np.std(win_aucs):.4f}")
    print(f"  Best:  {max(win_aucs):.4f}  |  Worst: {min(win_aucs):.4f}")

    # Plot
    fig,axes=plt.subplots(1,2,figsize=(14,4))
    colors=['#2ecc71' if a>0.5 else '#e74c3c' for a in win_aucs]
    axes[0].bar(range(1,n+1),win_aucs,color=colors,edgecolor='k',lw=0.5)
    axes[0].axhline(0.5,color='k',ls='--',label='Chance')
    axes[0].axhline(np.mean(win_aucs),color='blue',ls='-',lw=2,label=f'Mean={np.mean(win_aucs):.3f}')
    axes[0].set(title='Window-AUC per Fold',xlabel='Fold',ylabel='AUC',ylim=(0.4,1.0)); axes[0].legend()

    avg=np.mean([r['aucs'] for r in fold_results],axis=0)
    std=np.std( [r['aucs'] for r in fold_results],axis=0)
    t_ms=np.arange(len(avg))*(1000/FS)
    axes[1].fill_between(t_ms,avg-std,avg+std,alpha=0.25,color='blue')
    axes[1].plot(t_ms,avg,'b-',lw=2,label='Mean window-AUC')
    axes[1].axhline(0.5,color='red',ls='--',lw=1.5)
    axes[1].axvspan(300,900,alpha=0.08,color='green',label='Signal [300-900ms]')
    axes[1].set(title='AUC Time-Course',xlabel='Time (ms)',ylabel='AUC'); axes[1].legend()
    plt.suptitle(f'LOSO CV — Mean Window-AUC = {np.mean(win_aucs):.4f}',fontweight='bold')
    plt.tight_layout(); plt.show()
    return fold_results

print("✓ LOSO defined")


## Cell 13 — Final Training & Submission

**Fixed vs v5.0:**
- OOF loop now correctly indexes by `n_trials` (not timepoints)
- Uses pre-cached `X_all` (no re-extraction per OOF fold)
- CSP inside OOF is fitted only on fold's training subjects
- Isotonic fitted on `(n_train_subjects,)` dimension correctly


In [ ]:
def generate_submission(train_subjects, test_subjects, output_path=OUTPUT):
    print("\n"+"="*66)
    print("  Final Training → Submission Generation")
    print("="*66)

    # ── 1. Extract ALL training features (once) ────────────────────────────
    print("\nStep 1/5: Extracting training features...")
    X_all, y_all, eeg_all, lbl_all = [], [], [], []
    n_trials_list = []
    for i,s in enumerate(train_subjects):
        t0=time.time()
        print(f"  [{i+1}/{len(train_subjects)}] {s['id']} ...", end=' ', flush=True)
        X,y,_,zmu,zsig = extract_subject(s)
        X_all.append(X); y_all.append(y)
        eeg_all.append(s['eeg']); lbl_all.append(s['labels'])
        n_trials_list.append(s['eeg'].shape[0])
        print(f"{X.shape}  ({time.time()-t0:.0f}s)")

    X_tr   = np.vstack(X_all)
    y_tr   = np.concatenate(y_all)
    eeg_tr = np.vstack(eeg_all)
    lbl_tr = np.concatenate(lbl_all)
    n_tr   = eeg_tr.shape[0]
    y_bin  = (lbl_tr==2).astype(int)
    print(f"\n  Total training: {X_tr.shape}  [emo={y_bin.sum()} neu={(1-y_bin).sum()}]")

    # ── 2. Fit global CSP ──────────────────────────────────────────────────
    print("\nStep 2/5: Fitting CSP...")
    csp=CSP(); csp.fit(eeg_tr,lbl_tr)
    csp_tr=csp.log_var(eeg_tr)
    X_tr_f=np.hstack([X_tr,csp_tr])
    print(f"  Feature matrix: {X_tr_f.shape}")

    # ── 3. OOF predictions for isotonic calibration ────────────────────────
    # oof_probs[i] = predicted proba for trial i at each timepoint
    # Shape: (n_tr, N_TP) where n_tr = total number of TRIALS (not timepoints)
    print("\nStep 3/5: Building OOF predictions for isotonic calibration...")
    oof_probs = np.full((n_tr, N_TP), 0.5, dtype=np.float64)

    # Build cumulative trial index boundaries (per subject, in TRIAL space)
    trial_bounds = np.concatenate([[0], np.cumsum(n_trials_list)])  # length = n_subjects+1

    for i,s in enumerate(train_subjects):
        # Trial slice for this subject
        sl = slice(int(trial_bounds[i]), int(trial_bounds[i+1]))
        n_val_trials = n_trials_list[i]

        # Training fold: all subjects except i
        tr_idx = [j for j in range(len(train_subjects)) if j != i]
        n_tr_i = sum(n_trials_list[j] for j in tr_idx)

        # Stack training features (already extracted — no re-extraction)
        X_tr_i   = np.vstack([X_all[j] for j in tr_idx])
        y_tr_i   = np.concatenate([y_all[j] for j in tr_idx])
        eeg_tr_i = np.vstack([eeg_all[j] for j in tr_idx])
        lbl_tr_i = np.concatenate([lbl_all[j] for j in tr_idx])

        # CSP fitted on fold's training subjects only (no leakage)
        csp_i  = CSP(); csp_i.fit(eeg_tr_i, lbl_tr_i)
        csp_tri = csp_i.log_var(eeg_tr_i)
        csp_vi  = csp_i.log_var(s['eeg'])          # validation subject CSP
        X_tr_if = np.hstack([X_tr_i, csp_tri])
        X_v_if  = np.hstack([X_all[i], csp_vi])    # val features (already extracted)

        # Train TRE on fold's training data
        tre_i = TRE()
        tre_i.fit(X_tr_if, y_tr_i, n_tr_i, verbose=False)

        # Predict val subject → (n_val_trials, N_TP)
        P_raw = tre_i.predict(X_v_if, n_val_trials)
        P_sm  = gate(smooth(P_raw))

        # Store in the correct trial slice
        oof_probs[sl] = P_sm
        print(f"  OOF [{i+1:2d}/{len(train_subjects)}] {s['id']:30s} "
              f"trials[{int(trial_bounds[i])}:{int(trial_bounds[i+1])}]  done")
        del tre_i, X_tr_if, X_v_if, eeg_tr_i, csp_tri, csp_vi; gc.collect()

    # ── 4. Fit isotonic calibration per timepoint ─────────────────────────
    # y_bin_per_trial: (n_tr,) — one binary label per trial
    y_bin_trials = (lbl_tr == 2).astype(int)  # (n_tr,)
    print("\nStep 4/5: Fitting isotonic calibrators per timepoint...")
    iso_models = []
    for t in range(N_TP):
        iso=IsotonicRegression(out_of_bounds='clip')
        try:
            iso.fit(oof_probs[:,t], y_bin_trials)
        except:
            iso=None
        iso_models.append(iso)
    print(f"  ✓ {sum(1 for m in iso_models if m)} / {N_TP} calibrators fitted")

    # ── 5. Train FINAL TRE on ALL data ────────────────────────────────────
    print("\nStep 5/5: Training final TRE on ALL training data...")
    tre_final=TRE()
    tre_final.fit(X_tr_f, y_tr, n_tr, verbose=True)

    # ── 6. Predict test subjects ───────────────────────────────────────────
    print("\nGenerating test predictions...")
    rows=[]
    for s in test_subjects:
        sid=s['subj_id']; n_te=s['eeg'].shape[0]
        print(f"\n  Subject {sid} ({s['id']}): {n_te} trials")

        X_te,_,_,_,_=extract_subject(s)
        csp_te=csp.log_var(s['eeg'])
        X_te_f=np.hstack([X_te,csp_te])

        P_raw=tre_final.predict(X_te_f,n_te)
        P_sm =smooth(P_raw)
        P_gat=gate(P_sm)

        # Apply isotonic calibration
        P_cal=np.zeros_like(P_gat)
        for t in range(N_TP):
            if iso_models[t]: P_cal[:,t]=iso_models[t].transform(P_gat[:,t])
            else:              P_cal[:,t]=P_gat[:,t]
        P_cal=np.clip(P_cal,0.01,0.99)

        print(f"    range:[{P_cal.min():.3f},{P_cal.max():.3f}]  mean:{P_cal.mean():.4f}")

        for trial_id in range(n_te):
            for tp in range(N_TP):
                rows.append({'id':f"{sid}_{trial_id}_{tp}",
                             'prediction':float(P_cal[trial_id,tp])})

    df=pd.DataFrame(rows)
    df.to_csv(output_path,index=False)
    print(f"\n{'='*66}")
    print(f"✓ Saved → {output_path}  ({len(df):,} rows)")
    print(df.head(6).to_string(index=False))
    return df

print("✓ generate_submission defined")


## ▶ Cell 14 — MAIN

In [ ]:
RUN_LOSO = False   # True = run LOSO CV first (~60-90 min), then submission

print("""
╔══════════════════════════════════════════════════════════════════╗
║   EEG Emotional Memory — GPU Ultra Pipeline v5.1               ║
╠══════════════════════════════════════════════════════════════════╣
║  TimeResolved: 121 classifiers × (LDA+LGBM+XGBoost)            ║
║  Features    : ~549/tp  (power+DE+PLV+cov+CSP)                 ║
║  Signal gate : train only on [300-900ms] window                 ║
║  OOF isotonic calibration per timepoint (FIXED)                 ║
╚══════════════════════════════════════════════════════════════════╝
""")

T0=time.time()

print("STEP 1: Loading training data...")
train_subjects=load_all_training(EMO_DIR,NEU_DIR)

print("\nSTEP 2: Loading test data...")
test_subjects=load_all_test(TEST_DIR)

if RUN_LOSO:
    print("\nSTEP 3: LOSO Cross-Validation...")
    fold_results=run_loso(train_subjects)
    win_aucs=[r['window_auc'] for r in fold_results]
    print(f"\n  ► Mean Window-AUC = {np.mean(win_aucs):.4f} ± {np.std(win_aucs):.4f}")
else:
    print("\nSTEP 3: Skipping LOSO  (set RUN_LOSO=True to enable)")
    fold_results=[]

print("\nSTEP 4: Final model + submission...")
df=generate_submission(train_subjects,test_subjects,OUTPUT)

print("\nSTEP 5: Validation...")
assert 'id' in df.columns and 'prediction' in df.columns
assert df['prediction'].between(0,1).all()
assert all(len(str(r).split('_'))==3 for r in df['id'].head(10))
print(f"  ✓ {len(df):,} rows  |  range [{df.prediction.min():.4f},{df.prediction.max():.4f}]")

# Download / copy
try:
    from google.colab import files; files.download(OUTPUT)
    print("  ✓ Download triggered (Colab)")
except:
    import shutil
    try: shutil.copy(OUTPUT,'./submission.csv'); print(f"  ✓ Copied to ./submission.csv")
    except: pass

ws=f"Mean Window-AUC={np.mean([r['window_auc'] for r in fold_results]):.4f}" if fold_results else "LOSO skipped"
print(f"""
╔══════════════════════════════════════════════════════════════════╗
║  ✓ DONE  ({(time.time()-T0)/60:.1f} min)                                     ║
║  {ws:<64s}║
║  Saved: {OUTPUT:<58s}║
╚══════════════════════════════════════════════════════════════════╝
""")


## Cell 15 — Diagnostics

In [ ]:
def diagnose(path):
    print(f"\nDiagnosing: {path}")
    with h5py.File(path,'r') as f:
        f.visititems(lambda n,o: print(
            f"  {n:45s}  shape={getattr(o,'shape','group')}  dtype={getattr(o,'dtype','—')}"))

# diagnose(f'{EMO_DIR}/S_10_cleaned.mat')
print("✓ diagnose() ready — uncomment above to use")
